In [2]:
# ==========================================================
# ACTIVIDAD 18. Perfiles de los candidatos k = 3 y k = 6
# Describe cada régimen en unidades reales, mira cómo se
# reparte en el tiempo y comprueba la estabilidad con varias
# semillas. Solo empresas de entrenamiento (bloques de
# entrenamiento y validación). Las reservadas NO se usan.
# ==========================================================
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

SEMILLA = 42
CANDIDATOS = [3, 6]
SEMILLAS_EXTRA = [1, 2, 3]          # para comprobar que la estabilidad no depende de la semilla
VARIABLES = ["volatilidad_21", "cambio_volatilidad", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]
COLUMNAS_Z = ["z_" + v for v in VARIABLES]

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Datos de las empresas de entrenamiento
# ----------------------------------------------------------
df = pd.read_parquet(RAIZ / "01_datos" / "variables_preparadas.parquet",
                     columns=["date", "grupo", "bloque"] + VARIABLES + COLUMNAS_Z)
df = df[(df["grupo"] == "entrenamiento") & (df["bloque"].isin(["entrenamiento", "validacion"]))].copy()
df["bloque"] = df["bloque"].astype(str)
es_ent = df["bloque"] == "entrenamiento"
X_ent = df.loc[es_ent, COLUMNAS_Z].to_numpy()
X_val = df.loc[~es_ent, COLUMNAS_Z].to_numpy()
print("Filas de entrenamiento:", len(X_ent), "| validación:", len(X_val))

perfiles = []
for k in CANDIDATOS:
    print("\n" + "=" * 70)
    print(f"CANDIDATO k = {k}")
    print("=" * 70)
    modelo = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_ent)

    # Etiquetas en ambos bloques y renumeración: régimen 1 = menor volatilidad
    etiquetas = np.empty(len(df), dtype=int)
    etiquetas[es_ent.to_numpy()] = modelo.labels_
    etiquetas[~es_ent.to_numpy()] = modelo.predict(X_val)
    orden = np.argsort(modelo.cluster_centers_[:, 0])          # columna 0 = z_volatilidad_21
    renumerar = {antiguo: nuevo + 1 for nuevo, antiguo in enumerate(orden)}
    df["regimen"] = pd.Series(etiquetas, index=df.index).map(renumerar)

    # a) Perfil de cada régimen en unidades reales (medianas, bloque de entrenamiento)
    ent = df[es_ent]
    perfil = ent.groupby("regimen")[VARIABLES].median()
    perfil.insert(0, "pct_filas", ent["regimen"].value_counts(normalize=True).sort_index() * 100)
    print("\nPerfil de cada régimen (medianas en el bloque de entrenamiento):")
    print(perfil)

    # b) Centros en unidades escaladas (cuántas desviaciones por encima o debajo de la media)
    centros = pd.DataFrame(modelo.cluster_centers_[orden], columns=VARIABLES,
                           index=pd.Index(range(1, k + 1), name="regimen"))
    print("\nCentros en unidades escaladas (0 = media del entrenamiento):")
    print(centros)

    # c) Proporción de cada régimen en entrenamiento y en validación
    print("\n% de filas por régimen y bloque:")
    print((pd.crosstab(df["regimen"], df["bloque"], normalize="columns") * 100))

    # d) Estabilidad con otras semillas para el modelo entrenado en validación
    etiquetas_val = modelo.predict(X_val)
    aris = [adjusted_rand_score(
                KMeans(n_clusters=k, n_init=10, random_state=s).fit(X_val).labels_, etiquetas_val)
            for s in SEMILLAS_EXTRA]
    print("\nARI entrenamiento vs validación con semillas", SEMILLAS_EXTRA, ":", np.round(aris, 3))

    perfil["k"] = k
    perfiles.append(perfil.reset_index())

    # e) Evolución trimestral (solo k = 3, para que sea legible)
    if k == 3:
        trimestre = df["date"].dt.to_period("Q")
        print("\n% de filas por régimen en cada trimestre (k = 3):")
        print(pd.crosstab(trimestre, df["regimen"], normalize="index") * 100)

pd.concat(perfiles).to_csv(RAIZ / "02_modelos" / "perfiles_candidatos.csv", index=False)
print("\nGuardado: 02_modelos/perfiles_candidatos.csv")

Mounted at /content/drive
Filas de entrenamiento: 815829 | validación: 488607

CANDIDATO k = 3

Perfil de cada régimen (medianas en el bloque de entrenamiento):
         pct_filas  volatilidad_21  cambio_volatilidad  rendimiento_21  caida_maximo_63  volumen_anormal
regimen                                                                                                 
1           52.909           0.307              -0.167           0.012           -0.080           -0.080
2           32.921           0.409               0.105           0.051           -0.060            0.334
3           14.170           0.807               0.126          -0.180           -0.345            0.006

Centros en unidades escaladas (0 = media del entrenamiento):
         volatilidad_21  cambio_volatilidad  rendimiento_21  caida_maximo_63  volumen_anormal
regimen                                                                                      
1                -0.425              -0.517           0.071     